# Session 1 — Human lymph-node input and RNA quality control

## Goal

Follow one transcript-only human lymph-node section from its H&E image and manual germinal-center (GC) labels through spot-level RNA QC. DGAT predictions can deteriorate when transcript coverage is low, so detected genes per spot are a model-input diagnostic, not merely a cosmetic filter.

Session 2 will load organizer-validated predictions for the exact spot set retained here.

In [ ]:
REQUIREMENTS = [('anndata', 'anndata==0.11.4'), ('scanpy', 'scanpy==1.11.5')]

from pathlib import Path
import importlib, importlib.util, json, os, shutil, subprocess, sys

SESSION_REQUIREMENTS = REQUIREMENTS
in_colab = importlib.util.find_spec("google.colab") is not None and Path("/content").is_dir()
if in_colab:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    repo_dir = Path("/content/ECCB-2026-Tutorial")
    tutorial_root = repo_dir / "hands-on_tutorial"
    if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"], check=True)
    drive_root = Path("/content/drive/MyDrive/ECCB2026")
    manifest_path = drive_root / "asset_manifest.json"
    if not manifest_path.is_file(): raise FileNotFoundError(f"Missing {manifest_path}; run Session 0.")
    manifest = json.loads(manifest_path.read_text())
    drive_assets = drive_root / "assets" / "DGAT_assets"
    local_assets = tutorial_root / "external" / "DGAT_assets"
    local_data = local_assets / "data"
    local_data.mkdir(parents=True, exist_ok=True)
    for filename in ("V1_Human_Lymph_Node_filtered_feature_bc_matrix.h5", "V1_Human_Lymph_Node_manual_GC_annot.csv"):
        source, destination = drive_assets / "data" / filename, local_data / filename
        expected = manifest["files"][filename]["bytes"]
        if not source.is_file() or source.stat().st_size != expected: raise IOError(f"Invalid Drive asset: {source}")
        if not destination.is_file() or destination.stat().st_size != expected: shutil.copy2(source, destination)
    source_spatial, destination_spatial = drive_assets / "data" / "spatial", local_data / "spatial"
    if not source_spatial.is_dir(): raise FileNotFoundError(f"Missing {source_spatial}")
    if destination_spatial.exists(): shutil.rmtree(destination_spatial)
    shutil.copytree(source_spatial, destination_spatial)
    os.environ["DGAT_TUTORIAL_STATE_DIR"] = str(drive_root / "state")
    dgat_dir = tutorial_root / "external" / "DGAT"
    if not (dgat_dir / "utils" / "Preprocessing.py").is_file():
        dgat_dir.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/DGAT.git", str(dgat_dir)], check=True)
    missing = [spec for module, spec in SESSION_REQUIREMENTS if importlib.util.find_spec(module) is None]
    if missing:
        wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
        if wheelhouse.is_dir(): cmd[4:4] = ["--find-links", str(wheelhouse)]
        subprocess.run(cmd, check=True); importlib.invalidate_caches()
else:
    tutorial_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "dgat_tutorial").is_dir()), None)
    if tutorial_root is None: raise FileNotFoundError("Could not locate hands-on_tutorial.")
    drive_root = None

os.chdir(tutorial_root)
sys.path.insert(0, str(tutorial_root / "src"))
from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint
paths = tutorial_paths(tutorial_root)
print("Tutorial root:", paths.root)
print("Completed checkpoints:", sorted(p.name for p in paths.checkpoints.glob("session_*/part_*.json")) or "none")


## Step 1 — Load and align the sample

The 10x sample contains RNA counts and spatial coordinates, but no measured proteins. Alignment is explicit: unmatched annotation barcodes are reported rather than silently interpreted as tissue spots.


In [ ]:
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from dgat_tutorial.data import load_tutorial_data, load_gc_annotations
from dgat_tutorial.processing import calculate_rna_qc_metrics, DGAT_MIN_GENES

dataset = load_tutorial_data(paths.root / "external/DGAT_assets/data")
gc_path = paths.root / "external/DGAT_assets/data/V1_Human_Lymph_Node_manual_GC_annot.csv"
gc_positive = load_gc_annotations(gc_path)
spots, transcripts = dataset.spots.copy(), dataset.transcripts.copy()
positive_in_tissue = spots.index.intersection(gc_positive.index, sort=False)
annotation_without_tissue = gc_positive.index.difference(spots.index)
print({"tissue_spots": len(spots), "gc_positive_annotations": len(gc_positive),
       "gc_positive_in_tissue": len(positive_in_tissue),
       "annotation_without_tissue": len(annotation_without_tissue)})
gc = pd.Series(spots.index.isin(gc_positive.index), index=spots.index, name="germinal_center")
spots["germinal_center"] = gc
assert spots.index.equals(transcripts.index) and spots.index.equals(gc.index)
display(spots.head(3))


## Step 2 — Inspect tissue morphology and GC labels

The H&E image provides morphological context; colored spots show the manual GC reference used later. A GC label is an anatomical annotation, not a measured protein value.


In [ ]:
from PIL import Image
image = np.asarray(Image.open(dataset.image_path))
scale = dataset.scale_factors.get("tissue_hires_scalef", 1.0)
fig, ax = plt.subplots(figsize=(8, 7)); ax.imshow(image)
other = ~gc; ax.scatter(spots.loc[other,"x"]*scale, spots.loc[other,"y"]*scale, s=5, c="#9e9e9e", alpha=.25, label="Other")
ax.scatter(spots.loc[gc,"x"]*scale, spots.loc[gc,"y"]*scale, s=9, c="#d73027", alpha=.8, label="GC")
ax.set(title="Human lymph node: H&E and manual germinal centers"); ax.axis("off"); ax.legend()
he_path = paths.figures / "session01_lymph_node_he_gc.png"; fig.savefig(he_path, dpi=170, bbox_inches="tight"); plt.show()


## Step 3 — Calculate RNA QC and model-compatible coverage

Total UMIs and all detected genes describe the raw library. DGAT's released `preprocess_ST` filter is applied *after* alignment to its 11,535 input genes, so we also count detected genes within that exact model contract. This second count determines which spots can align to the precomputed prediction matrix.

In [ ]:
qc = calculate_rna_qc_metrics(transcripts)
common_genes = [x.strip() for x in (paths.root / "external/DGAT/resources/common_gene_11535.txt").read_text().splitlines() if x.strip()]
available_model_genes = [gene for gene in common_genes if gene in transcripts.columns]
qc["dgat_model_genes_detected"] = transcripts.loc[:, available_model_genes].gt(0).sum(axis=1).astype(int)

fig, axes = plt.subplots(1,4,figsize=(16,3.5))
for ax, col, label in zip(
    axes,
    ["rna_total","genes_detected","dgat_model_genes_detected","mt_pct"],
    ["Total UMIs","All detected genes","Detected DGAT input genes","Mitochondrial counts (%)"],
):
    ax.hist(qc[col], bins=40, color="#4c78a8"); ax.set(xlabel=label,ylabel="spots")
axes[2].axvline(DGAT_MIN_GENES,color="#d73027",ls="--",label="DGAT minimum = 700"); axes[2].legend()
qc_path = paths.results / "session01_rna_qc.csv"; qc.to_csv(qc_path)
qc_fig_path = paths.figures / "session01_rna_qc.png"; fig.tight_layout(); fig.savefig(qc_fig_path,dpi=160); plt.show()

## Step 4 — Map model-compatible coverage and create the handoff

Spatially localized low coverage can create spatially localized model uncertainty. The map should therefore be considered when interpreting every downstream pattern. The retained barcode order is the hard contract checked against the precomputed predictions in Session 2.

In [ ]:
from dgat_tutorial.plotting import plot_spatial_feature
ax = plot_spatial_feature(spots, qc["dgat_model_genes_detected"], "Detected DGAT input genes per spot", cmap="magma")
coverage_path = paths.figures / "session01_detected_dgat_genes_spatial.png"; plt.savefig(coverage_path,dpi=160,bbox_inches="tight"); plt.show()
keep = qc["dgat_model_genes_detected"] >= DGAT_MIN_GENES
filtered_spots, filtered_rna, filtered_qc, filtered_gc = spots.loc[keep].copy(), transcripts.loc[keep].copy(), qc.loc[keep].copy(), gc.loc[keep].copy()
if not (filtered_qc["dgat_model_genes_detected"] >= DGAT_MIN_GENES).all(): raise AssertionError("DGAT model-gene threshold failed")
from scipy import sparse
filtered_spots.to_csv(paths.processed_data / "filtered_spots.csv")
sparse.save_npz(paths.processed_data / "filtered_rna_counts.npz", sparse.csr_matrix(filtered_rna.to_numpy(dtype=np.float32)), compressed=True)
pd.Series(filtered_rna.columns, name="gene").to_csv(paths.processed_data / "filtered_rna_genes.csv", index=False)
corresponding_genes = [gene for gene in [p.split("_")[0] for p in [x.strip() for x in (paths.root / "external/DGAT/resources/common_protein_31.txt").read_text().splitlines() if x.strip()]] if gene in filtered_rna.columns]
filtered_rna.loc[:, corresponding_genes].to_csv(paths.processed_data / "corresponding_rna_counts.csv")
filtered_qc.to_csv(paths.processed_data / "rna_qc.csv")
filtered_gc.rename("germinal_center").to_csv(paths.processed_data / "germinal_center_labels.csv")
summary = pd.DataFrame([{"sample":"V1_Human_Lymph_Node","input_spots":len(spots),"retained_spots":int(keep.sum()),"removed_below_700_model_genes":int((~keep).sum()),"observed_genes":filtered_rna.shape[1],"dgat_contract_genes":len(common_genes),"gc_spots":int(filtered_gc.sum())}])
summary_path = paths.results / "session01_qc_summary.csv"; summary.to_csv(summary_path,index=False); display(summary)
manifest = write_checkpoint("1.1", [qc_path,qc_fig_path,he_path,coverage_path,summary_path,paths.processed_data/"filtered_spots.csv",paths.processed_data/"filtered_rna_counts.npz",paths.processed_data/"filtered_rna_genes.csv",paths.processed_data/"corresponding_rna_counts.csv",paths.processed_data/"germinal_center_labels.csv"], summary=summary.iloc[0].to_dict(), start=paths.root)
print("Checkpoint:",manifest)

## Check

Confirm that all four objects—RNA, coordinates, QC, and GC labels—have identical ordered spot IDs. Explain why a spot passing 700 detected genes can still have less reliable predictions than a deeply sequenced spot.


## Next steps

Session 2 loads the validated precomputed matrix and requires its barcodes to be identical to this retained spot index. No measured lymph-node protein values are introduced.